# Session 1: boxes, pipes and a loop
**A. Real-world problem.** In a chemical plant a stream of liquid goes into a separator, and
the part that is not good enough is sent back to the start to try again. Every box obeys a
simple rule ("what goes in comes out"), but because of the loop the flow into the first box
depends on the flow out of the last. How much liquid really flows through the separator?
Commercial flowsheet programs answer this for plants with hundreds of boxes; today you
do it for two, in the spirit of those tools but at learning scale.

Words: a **stream** is a pipe with a flow (mol/s). A **box** has inlets, outlets and a **rule**.
The **recycle** is the stream that goes back to the start.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: write each box as one equation
Streams: F = fresh feed (known), S = flow into the separator, P = product, R = recycle.
* Mixer box (joins F and R):          S = F + R           ->  S - R = F
* Separator box, rule 1 (recycle):    R = r * S           ->  R - r S = 0
* Separator box, rule 2 (product):    P = (1 - r) * S     ->  P - (1 - r) S = 0
r is the **recycle fraction** (0 to 1). Three unknowns (S, P, R), three equations: solvable.

In [ ]:
def plant_equations(F, r):
    """Matrix A and right-hand side b of A @ [S, P, R] = b for the mixer and separator rules."""
    A = np.array([[1.0, 0.0, -1.0],
                  [-r, 0.0, 1.0],
                  [-(1.0 - r), 1.0, 0.0]])
    b = np.array([F, 0.0, 0.0])
    return A, b


def solve_plant(F, r):
    """Solve all box rules together. Returns dict with S, P, R in mol/s."""
    if not 0.0 <= r < 1.0:
        raise ValueError("recycle fraction must be at least 0 and below 1: with r = 1 nothing can leave the plant")
    A, b = plant_equations(F, r)
    S, P, R = np.linalg.solve(A, b)
    return {"S": float(S), "P": float(P), "R": float(R)}

## Step 2: the "guess and go round" method
Another way: guess the recycle, run the mixer and separator in order, and see what recycle
comes back. If it differs from the guess, use the new value and go round again. This is
**successive substitution**, and it is how flowsheet programs treat a loop (Session 3).

In [ ]:
def go_round(F, r, tol=1e-6, max_rounds=10000):
    """Successive substitution on the recycle. Returns (R, rounds, list of R after each round)."""
    R, hist = 0.0, []
    for k in range(1, max_rounds + 1):
        S = F + R               # mixer
        R_new = r * S           # separator sends r*S back
        hist.append(R_new)
        if abs(R_new - R) < tol:
            return R_new, k, hist
        R = R_new
    return R, max_rounds, hist


def amplification(r):
    """S / F = 1 / (1 - r): how many times the fresh feed flows through the separator."""
    return 1.0 / (1.0 - r)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    out = solve_plant(100.0, 0.6)
    print("F = 100, r = 0.6 ->", {k: round(v, 1) for k, v in out.items()})
    R, rounds, hist = go_round(100.0, 0.6)
    print("rounds needed:", rounds, " first guesses:", [round(h, 1) for h in hist[:3]])
    print("r = 0.9: S =", round(solve_plant(100.0, 0.9)["S"], 1), " rounds:", go_round(100.0, 0.9)[1])

    rs = np.linspace(0, 0.95, 96)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 4.0))
    a1.plot(rs, [amplification(r) for r in rs])
    a1.set_xlabel("recycle fraction r (fraction of S sent back)")
    a1.set_ylabel("S / F (flow through separator per unit feed)")
    a1.set_title("A bigger recycle needs a bigger separator")
    a1.grid(alpha=0.3)
    for r in (0.5, 0.8, 0.9):
        _, _, h = go_round(100.0, r, max_rounds=60)
        a2.plot(range(1, len(h) + 1), h, label=f"r = {r}")
    a2.set_xlabel("round number")
    a2.set_ylabel("recycle guess R (mol/s)")
    a2.set_title("Going round: R creeps to its final value")
    a2.legend(fontsize=8)
    a2.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "loop.png", dpi=120)

## Where this stops
One number per pipe, a loop that only splits. Session 2 puts real physics in the boxes
(temperature, composition, boiling), and Session 3 builds the engine that solves a whole flowsheet.

## Coding tasks
1. **Easy.** Change the recycle fraction in `lesson.py` to 0.75 and predict S, R and P before you run it.
2. **Stretch.** Add a function `max_recycle(F, S_max)` that returns the largest r allowed by a separator limit, and test it with F = 150 and S_max = 400.
3. **Explain.** Explain in your own words why the product flow does not grow when you increase the recycle fraction, but the separator size does.

## Assessment
- **A. Conceptual.** Why is the product flow equal to the fresh feed even though the separator handles much more?
- **B. Numerical.** Find S, R and P for F = 100 mol/s and r = 0.75.
- **C. Graph and simulation interpretation.** On the left figure, what does the separator handle per unit feed at r = 0.8 and at r = 0.95, and what does that mean for equipment size?
- **D. Design and optimisation challenge.** Design: the separator can pass at most 400 mol/s. What is the largest recycle fraction at a feed of 100 mol/s, and what happens if the feed is raised to 150 mol/s? What would you trade away?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    out = L.solve_plant(100.0, 0.6)
    checks.append((f"worked example: F = 100, r = 0.6 gives S = 250, R = 150, P = 100 (got {out['S']:.1f}, {out['R']:.1f}, {out['P']:.1f})",
                   abs(out["S"] - 250.0) < 1e-9 and abs(out["R"] - 150.0) < 1e-9 and abs(out["P"] - 100.0) < 1e-9))
    checks.append(("hand calculation: S = F / (1 - r) = 100 / 0.4 = 250", abs(100.0 / 0.4 - out["S"]) < 1e-9 and abs(L.amplification(0.6) - 2.5) < 1e-12))
    checks.append(("overall balance: product equals fresh feed at steady state", abs(out["P"] - 100.0) < 1e-9))
    A, b = L.plant_equations(100.0, 0.6)
    checks.append(("3 unknowns and 3 equations (square system)", A.shape == (3, 3) and b.shape == (3,)))
    checks.append(("the solution satisfies every box rule (residual below 1e-9)", float(np.max(np.abs(A @ np.array([out["S"], out["P"], out["R"]]) - b))) < 1e-9))
    R, rounds, hist = L.go_round(100.0, 0.6)
    checks.append((f"guess and go round: first guesses 60, 96, 117.6 (got {hist[0]:.1f}, {hist[1]:.1f}, {hist[2]:.1f})",
                   abs(hist[0] - 60.0) < 1e-9 and abs(hist[1] - 96.0) < 1e-9 and abs(hist[2] - 117.6) < 1e-9))
    checks.append((f"going round reaches R = 150 in 37 rounds at tolerance 1e-6 (got {rounds})", rounds == 37 and abs(R - 150.0) < 1e-4))
    checks.append(("going round and solving together agree (r = 0.5, 0.8)",
                   all(abs(L.go_round(100.0, r, tol=1e-10)[0] - L.solve_plant(100.0, r)["R"]) < 1e-6 for r in (0.5, 0.8))))
    s9 = L.solve_plant(100.0, 0.9)
    checks.append((f"r = 0.9: separator flow 1000 mol/s, ten times the feed (got {s9['S']:.0f})", abs(s9["S"] - 1000.0) < 1e-6 and abs(L.amplification(0.9) - 10.0) < 1e-9))
    checks.append((f"r = 0.9 needs 175 rounds at tolerance 1e-6 (got {L.go_round(100.0, 0.9)[1]})", L.go_round(100.0, 0.9)[1] == 175))
    r0 = L.solve_plant(100.0, 0.0)
    checks.append(("edge: r = 0 means no recycle, S = P = 100 and R = 0", abs(r0["S"] - 100.0) < 1e-12 and r0["R"] == 0.0 and abs(r0["P"] - 100.0) < 1e-12))
    try:
        L.solve_plant(100.0, 1.0)
        bad = False
    except ValueError:
        bad = True
    checks.append(("edge: r = 1 is refused with a clear message (nothing could leave)", bad))
    checks.append(("flows scale with the feed: F = 250 gives S = 625 at r = 0.6", abs(L.solve_plant(250.0, 0.6)["S"] - 625.0) < 1e-9))
    checks.append(("figure values: S/F is 2, 5 and 20 at r = 0.5, 0.8, 0.95; R is 100, 400, 900 at r = 0.5, 0.8, 0.9",
                   [round(L.amplification(r), 6) for r in (0.5, 0.8, 0.95)] == [2.0, 5.0, 20.0]
                   and [round(L.solve_plant(100.0, r)["R"], 6) for r in (0.5, 0.8, 0.9)] == [100.0, 400.0, 900.0]))
    q = L.solve_plant(100.0, 0.75)
    checks.append(("exercise: F = 100 and r = 0.75 give S = 400, R = 300, P = 100", abs(q["S"] - 400) < 1e-9 and abs(q["R"] - 300) < 1e-9 and abs(q["P"] - 100) < 1e-9))
    checks.append(("design: a separator limited to 400 mol/s allows r = 1 - 100/400 = 0.75 at F = 100 and r = 0.625 at F = 150",
                   abs(L.solve_plant(150.0, 0.625)["S"] - 400.0) < 1e-9 and abs(1 - 100 / 400 - 0.75) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
